# Voting Regressor — Ensemble of Diverse Models
A **Voting Regressor** combines multiple different regression models and averages their predictions.

### Why it works
- Each model has different strengths and weaknesses
- Linear model captures global trends; tree captures local patterns; SVR handles non-linearity
- Averaging reduces individual model errors — errors tend to cancel out
- Result is often better than any single model alone

### Dataset — California Housing
8 features, target = median house value.


## Step 1: Imports and Load Dataset

In [2]:
from sklearn.datasets import fetch_california_housing
import numpy as np

In [3]:
housing = fetch_california_housing()
X, y = housing.data, housing.target

In [4]:
X.shape

(20640, 8)

In [5]:
y.shape

(20640,)

In [6]:
X

array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]], shape=(20640, 8))

## Step 2: Define Base Estimators
Three diverse regressors:
- `LinearRegression` — fast, interpretable, linear only
- `DecisionTreeRegressor` — captures non-linearity, prone to overfitting
- `SVR` — kernel-based, handles complex boundaries, slow on large data


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score

In [8]:
lr = LinearRegression()
dt = DecisionTreeRegressor()
svr = SVR()

## Step 3: Cross-Validate Each Model Individually
10-fold cross-validation with R² scoring gives a reliable performance baseline for each model.
Note which model performs best — the Voting Regressor should beat all of them.


In [9]:
estimators = [('lr',lr),('dt',dt),('svr',svr)]

## Step 4: Imports — VotingRegressor

In [10]:
for estimator in estimators:
  scores = cross_val_score(estimator[1],X,y,scoring='r2',cv=10)
  print(estimator[0],np.round(np.mean(scores),2))

lr 0.51
dt 0.26
svr -0.25


## Step 5: Equal-Weight Voting Regressor
Final prediction = simple **average** of all three models' predictions.
Compare the Voting Regressor R² against each individual model's R².


In [11]:
from sklearn.ensemble import VotingRegressor

## Step 6: Weighted Voting — Find Best Weights
Try all combinations of weights `[1,2,3]` for each model (27 combinations).
A higher weight gives that model more influence in the final average.

- `weights=[i, j, k]` → `prediction = (i×lr + j×dt + k×svr) / (i+j+k)`

Watch for which weight combination gives the highest R².


In [12]:
vr = VotingRegressor(estimators)
scores = cross_val_score(vr,X,y,scoring='r2',cv=10)
print("Voting Regressor",np.round(np.mean(scores),2))

Voting Regressor 0.47


---
## Part 2: Voting with Same Algorithm, Different Hyperparameters
Use 5 Decision Trees of increasing depth and combine them.

### Why this works
- Shallow trees (depth=1,2) → high bias, low variance → capture broad patterns
- Deep trees (depth=None) → low bias, high variance → capture fine detail
- Averaging them balances bias and variance better than any single depth


In [13]:
for i in range(1,4):
  for j in range(1,4):
    for k in range(1,4):
      vr = VotingRegressor(estimators,weights=[i,j,k])
      scores = cross_val_score(vr,X,y,scoring='r2',cv=10)
      print("For i={},j={},k={}".format(i,j,k),np.round(np.mean(scores),2))


For i=1,j=1,k=1 0.47
For i=1,j=1,k=2 0.37
For i=1,j=1,k=3 0.28
For i=1,j=2,k=1 0.48
For i=1,j=2,k=2 0.42
For i=1,j=2,k=3 0.36
For i=1,j=3,k=1 0.46
For i=1,j=3,k=2 0.44
For i=1,j=3,k=3 0.4
For i=2,j=1,k=1 0.51
For i=2,j=1,k=2 0.44
For i=2,j=1,k=3 0.37
For i=2,j=2,k=1 0.52
For i=2,j=2,k=2 0.47
For i=2,j=2,k=3 0.42
For i=2,j=3,k=1 0.51
For i=2,j=3,k=2 0.48
For i=2,j=3,k=3 0.45
For i=3,j=1,k=1 0.53
For i=3,j=1,k=2 0.47
For i=3,j=1,k=3 0.41
For i=3,j=2,k=1 0.54
For i=3,j=2,k=2 0.5
For i=3,j=2,k=3 0.45
For i=3,j=3,k=1 0.53
For i=3,j=3,k=2 0.51
For i=3,j=3,k=3 0.47


In [18]:
# using the same algorithm

dt1 = DecisionTreeRegressor(max_depth=1)
dt2 = DecisionTreeRegressor(max_depth=3)
dt3 = DecisionTreeRegressor(max_depth=5)
dt4 = DecisionTreeRegressor(max_depth=7)
dt5 = DecisionTreeRegressor(max_depth=None)

## Step 7: Cross-Validate Each Tree Depth
Compare R² for each individual tree depth. Notice the trade-off:
- Very shallow → underfits (low R²)
- Very deep → overfits (low test R²)
- Ensemble of all depths → beats the best individual


In [19]:
estimators = [('dt1',dt1),('dt2',dt2),('dt3',dt3),('dt4',dt4),('dt5',dt5)]

## Step 8: Voting Regressor (same algorithm, varied depth)
Combine all 5 trees into one Voting Regressor.
The ensemble R² should exceed all individual tree R² scores.


In [20]:
for estimator in estimators:
  scores = cross_val_score(estimator[1],X,y,scoring='r2',cv=10)
  print(estimator[0],np.round(np.mean(scores),2))

dt1 0.13
dt2 0.36
dt3 0.43
dt4 0.47
dt5 0.24


---
## Summary

| Setup | Expected Result |
|-------|----------------|
| Individual models (LR, DT, SVR) | Each has weaknesses |
| Equal-weight voting | Usually beats best individual |
| Weighted voting | Fine-tune by giving more weight to better models |
| Same algorithm, varied hyperparams | Diversity via depth variation |

**Key insight:** Voting works best when base models are **diverse and uncorrelated**.
If all models make the same errors, averaging doesn't help.
